# LeetCode 1836 — Remove Duplicates From an Unsorted Linked List bằng Python

Link đề: https://leetcode.com/problems/remove-duplicates-from-an-unsorted-linked-list/ · Độ khó: **Medium** · Chủ đề: Hash Table, Linked List

> **Ghi chú (bài Premium 🔒):** đề trong notebook này được viết lại theo hiểu biết về bài và theo bản mô tả công khai của kho doocs/leetcode. Nếu bạn có tài khoản LeetCode Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def deleteDuplicatesUnsorted(self, head: ListNode) -> ListNode:
        ...
```

## 1. Mục tiêu học

1. Tự cài `ListNode` và các hàm **dựng / xuất** danh sách liên kết kiểu LeetCode (`[1,2,3] ↔ 1 → 2 → 3`).
2. Phân biệt bài này (xóa **mọi** nút có giá trị xuất hiện ≥ 2 lần) với "giữ lại một bản" (LeetCode 83/82) và với danh sách **đã sắp**.
3. Dùng **hai lượt duyệt**: lượt 1 đếm tần suất (`Counter` hoặc mảng đếm), lượt 2 xóa nút bằng **nút giả** (dummy) và con trỏ `prev`.
4. So sánh với vét cạn `O(n²)`, cách "chuyển sang mảng rồi dựng lại", và cách dùng sắp xếp để tìm giá trị trùng.
5. Kiểm thử bằng danh sách ngẫu nhiên, kiểm tra cả việc **giữ nguyên thứ tự** các nút còn lại và xóa đúng ở **đầu/cuối** danh sách.

## 2. Đề bài & ví dụ

Cho `head` của một danh sách liên kết đơn (**không** sắp xếp). Tìm mọi giá trị xuất hiện **nhiều hơn một lần** trong danh sách và **xóa tất cả** các nút mang những giá trị đó. Trả về danh sách sau khi xóa (giữ nguyên thứ tự các nút còn lại).

**Ví dụ 1:** `head = [1,2,3,2]` → `[1,3]` (2 xuất hiện hai lần ⇒ xóa mọi nút 2).

**Ví dụ 2:** `head = [2,1,1,2]` → `[]` (cả 1 và 2 đều lặp ⇒ xóa hết).

**Ví dụ 3:** `head = [3,2,2,1,3,2,4]` → `[1,4]` (3 xuất hiện 2 lần, 2 xuất hiện 3 lần).

**Ràng buộc:** số nút trong `[1, 10⁵]`; `1 ≤ Node.val ≤ 10⁵`.

In [1]:
import random
import time
import sys
from collections import Counter
from typing import Optional, List

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build_list(arr):
    """Dựng danh sách liên kết từ list Python, trả về head (None nếu rỗng)."""
    dummy = ListNode(); cur = dummy
    for v in arr:
        cur.next = ListNode(v); cur = cur.next
    return dummy.next

def to_list(head):
    """Xuất danh sách liên kết thành list Python (có cầu chì chống vòng lặp)."""
    out, seen = [], 0
    while head:
        out.append(head.val); head = head.next; seen += 1
        if seen > 10 ** 6:
            raise RuntimeError("có chu trình?")
    return out

examples = [([1, 2, 3, 2], [1, 3]), ([2, 1, 1, 2], []), ([3, 2, 2, 1, 3, 2, 4], [1, 4])]
for arr, exp in examples:
    h = build_list(arr)
    print(f"head={to_list(h)} -> kỳ vọng {exp}")

head=[1, 2, 3, 2] -> kỳ vọng [1, 3]
head=[2, 1, 1, 2] -> kỳ vọng []
head=[3, 2, 2, 1, 3, 2, 4] -> kỳ vọng [1, 4]


## 3. Kiến thức nền

### 3.1 Danh sách liên kết đơn

Mỗi nút có `val` và `next`. Không truy cập ngẫu nhiên theo chỉ số được; muốn tới nút thứ `k` phải đi `k` bước từ `head`. Duyệt: `cur = head; while cur: …; cur = cur.next`.

In [2]:
h = build_list([3, 2, 2, 1])
cur, i = h, 0
while cur:
    print(f"nút {i}: val={cur.val}, next={'None' if cur.next is None else cur.next.val}")
    cur, i = cur.next, i + 1

nút 0: val=3, next=2
nút 1: val=2, next=2
nút 2: val=2, next=1
nút 3: val=1, next=None


### 3.2 Xóa một nút: nối `prev.next` bỏ qua nút đó

Muốn xóa `cur`, cần nút **ngay trước** nó (`prev`): `prev.next = cur.next`. Nút `cur` không còn được trỏ tới (bộ thu gom rác sẽ dọn).

In [3]:
h = build_list([1, 2, 3])
prev, cur = h, h.next           # xóa nút 2
prev.next = cur.next
print(to_list(h))

[1, 3]


### 3.3 Nút giả (dummy / sentinel)

Xóa nút **đầu** không có `prev` ⇒ phải xử lý riêng. Mẹo: tạo `dummy = ListNode(0, head)`; khi đó mọi nút thật đều có `prev`, kể cả nút đầu; cuối cùng trả `dummy.next`.

In [4]:
h = build_list([2, 1])
dummy = ListNode(0, h)
dummy.next = dummy.next.next      # xóa nút đầu (2) mà không cần trường hợp đặc biệt
print(to_list(dummy.next))

[1]


### 3.4 Đếm tần suất: `Counter` và mảng đếm

`Counter(values)` cho `{giá trị: số lần}`. Vì `1 ≤ val ≤ 10⁵` nên có thể dùng mảng `cnt = [0] * (10⁵ + 1)` — nhanh hơn dict trong Python.

In [5]:
vals = [3, 2, 2, 1, 3, 2, 4]
c = Counter(vals); print(c, "| trùng:", sorted(v for v, k in c.items() if k > 1))
cnt = [0] * 6
for v in vals: cnt[v] += 1
print("mảng đếm:", cnt)

Counter({2: 3, 3: 2, 1: 1, 4: 1}) | trùng: [2, 3]
mảng đếm: [0, 1, 3, 2, 1, 0]


### 3.5 In danh sách liên kết cho dễ nhìn

Khi trace thao tác con trỏ, in dạng `1 -> 2 -> 3 -> None` giúp thấy rõ nút nào đang bị nối/bỏ. Hàm `show` dưới đây còn đánh dấu nút mà một con trỏ đang trỏ tới.

In [6]:
def show(head, mark=None):
    parts, cur = [], head
    while cur:
        parts.append(f"[{cur.val}]" if cur is mark else str(cur.val)); cur = cur.next
    return " -> ".join(parts + ["None"])

h = build_list([3, 2, 2, 1])
print(show(h), "|", show(h, mark=h.next.next))

3 -> 2 -> 2 -> 1 -> None | 3 -> 2 -> [2] -> 1 -> None


## 4. Ý tưởng nền tảng / trực giác

Một nút bị xóa hay không phụ thuộc vào **toàn bộ** danh sách (giá trị đó có xuất hiện ở đâu khác không), kể cả các nút **phía sau**. Ví dụ `[3, 2, 2, 1, 3, 2, 4]`: khi đứng ở nút `3` đầu tiên, ta chưa biết có `3` khác ở vị trí 4 hay không ⇒ **không thể** quyết định trong một lượt duyệt đơn giản. ⇒ Cần **biết trước** tần suất ⇒ hai lượt.

Tính tay Ví dụ 3:

| giá trị | số lần | xóa? |
|---|---|---|
| 3 | 2 | có |
| 2 | 3 | có |
| 1 | 1 | không |
| 4 | 1 | không |

Duyệt lại, giữ các nút có tần suất 1: `1 → 4`.

In [7]:
arr = [3, 2, 2, 1, 3, 2, 4]
freq = Counter(arr)
print([(v, "giữ" if freq[v] == 1 else "xóa") for v in arr])

[(3, 'xóa'), (2, 'xóa'), (2, 'xóa'), (1, 'giữ'), (3, 'xóa'), (2, 'xóa'), (4, 'giữ')]


## 5. Cách 1 — Brute Force / baseline: với mỗi nút, đếm lại bằng cách duyệt cả danh sách

**Ý tưởng:** với mỗi nút `cur`, đi từ `head` đếm số nút có cùng giá trị (`O(n)`). Nếu đếm được `> 1` thì nút này cần xóa. Để không làm thay đổi số đếm trong lúc duyệt, ta **đánh dấu** trước, xóa sau.

In [8]:
def dedup_bruteforce(head):
    marks = []
    cur = head
    while cur:
        c, p = 0, head
        while p:
            c += p.val == cur.val; p = p.next
        marks.append(c > 1)
        cur = cur.next
    dummy = ListNode(0, head); prev, cur, i = dummy, head, 0
    while cur:
        if marks[i]: prev.next = cur.next
        else: prev = cur
        cur, i = cur.next, i + 1
    return dummy.next

for arr, exp in examples:
    got = to_list(dedup_bruteforce(build_list(arr))); print(got, "kỳ vọng", exp); assert got == exp

[1, 3] kỳ vọng [1, 3]
[] kỳ vọng []
[1, 4] kỳ vọng [1, 4]


**Độ phức tạp:** `O(n²)` thời gian (`10¹⁰` với `n = 10⁵`), `O(n)` cho mảng đánh dấu.

**Nút thắt:** số lần xuất hiện của mỗi giá trị bị tính lại cho **mỗi** nút cùng giá trị — trong khi chỉ cần tính **một lần** cho mỗi giá trị.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** tách thành hai lượt:
1. Lượt 1: đếm `freq[v]` cho mọi giá trị — `O(n)`.
2. Lượt 2: duyệt với `prev` (bắt đầu từ `dummy`) và `cur`:
   - **Nếu** `freq[cur.val] > 1` **thì** xóa: `prev.next = cur.next` (giữ nguyên `prev`).
   - **Ngược lại** giữ: `prev = cur`.
   - Luôn `cur = cur.next`.

**Invariant lượt 2:** danh sách từ `dummy` tới `prev` chỉ gồm các nút có tần suất 1, đúng thứ tự gốc; `prev.next` trỏ tới phần chưa xét (`cur`).

**Lưu ý:** khi xóa, **không** di chuyển `prev` — vì nút kế tiếp cũng có thể cần xóa (ví dụ `[2, 2]` liền nhau).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `Counter` + xóa tại chỗ với dummy

In [9]:
def dedup_counter(head):
    freq = Counter()
    cur = head
    while cur:
        freq[cur.val] += 1; cur = cur.next
    dummy = ListNode(0, head)
    prev, cur = dummy, head
    while cur:
        if freq[cur.val] > 1:
            prev.next = cur.next
        else:
            prev = cur
        cur = cur.next
    return dummy.next

print([to_list(dedup_counter(build_list(a))) for a, _ in examples])

[[1, 3], [], [1, 4]]


**Code nộp LeetCode (Cách 2):** xem mục 9. **Độ phức tạp:** Time `O(n)`, Space `O(u)` với `u` = số giá trị khác nhau.

### 7.2 Cách 3 — Mảng đếm cố định (giá trị `≤ 10⁵`)

Giống Cách 2 nhưng `freq` là `list` 100 001 phần tử — tránh băm, nhanh hơn trong Python khi `n` lớn.

In [10]:
class SolutionArray:
    def deleteDuplicatesUnsorted(self, head: ListNode) -> ListNode:
        cnt = [0] * 100001
        cur = head
        while cur:
            cnt[cur.val] += 1
            cur = cur.next
        dummy = ListNode(0, head)
        prev = dummy
        cur = head
        while cur:
            if cnt[cur.val] > 1:
                prev.next = cur.next
            else:
                prev = cur
            cur = cur.next
        return dummy.next

print([to_list(SolutionArray().deleteDuplicatesUnsorted(build_list(a))) for a, _ in examples])

[[1, 3], [], [1, 4]]


**Độ phức tạp Cách 3:** Time `O(n + V)` với `V = 10⁵` (khởi tạo mảng), Space `O(V)`.

### 7.3 Cách 4 — Chuyển sang mảng, lọc, dựng danh sách mới

Đơn giản nhất về tư duy: `vals = to_list(head)`, `keep = [v for v in vals if freq[v] == 1]`, rồi `build_list(keep)`. Đúng nhưng tạo **nút mới** (đề thường chấp nhận, nhưng nếu đề yêu cầu sửa tại chỗ hoặc nút có dữ liệu khác ngoài `val` thì không ổn).

In [11]:
def dedup_rebuild(head):
    vals = to_list(head)
    freq = Counter(vals)
    return build_list([v for v in vals if freq[v] == 1])

print([to_list(dedup_rebuild(build_list(a))) for a, _ in examples])

[[1, 3], [], [1, 4]]


**Độ phức tạp Cách 4:** Time `O(n)`, Space `O(n)` (mảng + nút mới).

### 7.4 Cách 5 — Tìm giá trị trùng bằng sắp xếp (không dùng bảng băm)

Sắp các giá trị, giá trị nào có phần tử kề bằng nó là giá trị trùng — thu thành tập `dup` (hoặc mảng cờ). Rồi xóa như Cách 2. Hữu ích khi không được dùng hash table (một số đề phỏng vấn cấm), đổi lại `O(n log n)`.

In [12]:
def dedup_sort(head):
    vals = sorted(to_list(head))
    dup = set()
    for i in range(1, len(vals)):
        if vals[i] == vals[i - 1]:
            dup.add(vals[i])
    dummy = ListNode(0, head); prev, cur = dummy, head
    while cur:
        if cur.val in dup: prev.next = cur.next
        else: prev = cur
        cur = cur.next
    return dummy.next

print([to_list(dedup_sort(build_list(a))) for a, _ in examples])

[[1, 3], [], [1, 4]]


**Độ phức tạp Cách 5:** Time `O(n log n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace lượt 2 của Cách 2 trên **Ví dụ 3** (`freq = {3:2, 2:3, 1:1, 4:1}`):

| bước | cur | freq | hành động | prev sau | danh sách (từ dummy) |
|---|---|---|---|---|---|
| 1 | 3 | 2 | xóa | dummy | D → 2 → 2 → 1 → … |
| 2 | 2 | 3 | xóa | dummy | D → 2 → 1 → … |
| 3 | 2 | 3 | xóa | dummy | D → 1 → 3 → 2 → 4 |
| 4 | 1 | 1 | giữ | 1 | D → 1 → 3 → 2 → 4 |
| 5 | 3 | 2 | xóa | 1 | D → 1 → 2 → 4 |
| 6 | 2 | 3 | xóa | 1 | D → 1 → 4 |
| 7 | 4 | 1 | giữ | 4 | D → 1 → 4 |

In [13]:
def trace_counter(arr):
    head = build_list(arr)
    freq = Counter(arr); print("freq =", dict(freq))
    dummy = ListNode(0, head); prev, cur, step = dummy, head, 0
    while cur:
        step += 1
        if freq[cur.val] > 1:
            prev.next = cur.next; act = "xóa"
        else:
            prev = cur; act = "giữ"
        print(f"bước {step}: cur={cur.val} (freq {freq[cur.val]}) -> {act:<3} | prev={'dummy' if prev is dummy else prev.val} | danh sách: {to_list(dummy.next)}")
        cur = cur.next
    return to_list(dummy.next)

assert trace_counter([3, 2, 2, 1, 3, 2, 4]) == [1, 4]

freq = {3: 2, 2: 3, 1: 1, 4: 1}
bước 1: cur=3 (freq 2) -> xóa | prev=dummy | danh sách: [2, 2, 1, 3, 2, 4]
bước 2: cur=2 (freq 3) -> xóa | prev=dummy | danh sách: [2, 1, 3, 2, 4]
bước 3: cur=2 (freq 3) -> xóa | prev=dummy | danh sách: [1, 3, 2, 4]
bước 4: cur=1 (freq 1) -> giữ | prev=1 | danh sách: [1, 3, 2, 4]
bước 5: cur=3 (freq 2) -> xóa | prev=1 | danh sách: [1, 2, 4]
bước 6: cur=2 (freq 3) -> xóa | prev=1 | danh sách: [1, 4]
bước 7: cur=4 (freq 1) -> giữ | prev=4 | danh sách: [1, 4]


Trace ví dụ xóa **toàn bộ** (Ví dụ 2) — `prev` luôn là `dummy`, kết quả `dummy.next = None`:

In [14]:
assert trace_counter([2, 1, 1, 2]) == []

freq = {2: 2, 1: 2}
bước 1: cur=2 (freq 2) -> xóa | prev=dummy | danh sách: [1, 1, 2]
bước 2: cur=1 (freq 2) -> xóa | prev=dummy | danh sách: [1, 2]
bước 3: cur=1 (freq 2) -> xóa | prev=dummy | danh sách: [2]
bước 4: cur=2 (freq 2) -> xóa | prev=dummy | danh sách: []


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — `Counter` + dummy, xóa tại chỗ.

In [15]:
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def deleteDuplicatesUnsorted(self, head: ListNode) -> ListNode:
        freq = Counter()
        cur = head
        while cur:
            freq[cur.val] += 1
            cur = cur.next
        dummy = ListNode(0, head)
        prev, cur = dummy, head
        while cur:
            if freq[cur.val] > 1:
                prev.next = cur.next
            else:
                prev = cur
            cur = cur.next
        return dummy.next

for arr, exp in examples:
    got = to_list(Solution().deleteDuplicatesUnsorted(build_list(arr))); print(got, exp); assert got == exp

[1, 3] [1, 3]
[] []
[1, 4] [1, 4]


## 10. Vì sao thuật toán đúng

Sau lượt 1, `freq[v]` = số nút mang giá trị `v` trong danh sách **gốc** (lượt 2 mới bắt đầu sửa, nên lượt 1 đếm đúng).

**Invariant lượt 2:** trước khi xét `cur`, (a) chuỗi `dummy → … → prev` gồm đúng các nút **đã xét** có `freq = 1`, theo thứ tự gốc; (b) `prev.next == cur`.
- Ban đầu `prev = dummy`, `cur = head`: (a) rỗng, (b) đúng.
- Nếu `freq[cur.val] > 1`: `prev.next = cur.next` ⇒ bỏ `cur` khỏi chuỗi; `prev` giữ nguyên ⇒ (a) vẫn đúng; sau `cur = cur.next` ⇒ (b) `prev.next == cur`.
- Ngược lại: `prev = cur` ⇒ thêm `cur` vào cuối chuỗi giữ lại ⇒ (a); `cur = cur.next` ⇒ (b) vì `prev.next` chưa đổi.

Khi `cur = None`, chuỗi `dummy.next …` chính là danh sách cần trả về (và `prev.next = None` kết thúc đúng). ∎

## 11. Phân tích Time / Space Complexity

Gọi `n` = số nút (`≤ 10⁵`), `u` = số giá trị khác nhau, `V = 10⁵` = giá trị lớn nhất.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Đếm lại cho mỗi nút | `O(n²)` | `O(n)` |
| 2. Counter + dummy | `O(n)` | `O(u)` |
| 3. Mảng đếm | `O(n + V)` | `O(V)` |
| 4. Chuyển mảng + dựng lại | `O(n)` | `O(n)` |
| 5. Sắp để tìm trùng | `O(n log n)` | `O(n)` |

In [16]:
rng = random.Random(1836)
arr = [rng.randint(1, 10 ** 5) for _ in range(10 ** 5)]
for name, f in [("Cách 2", Solution().deleteDuplicatesUnsorted), ("Cách 3", SolutionArray().deleteDuplicatesUnsorted),
                ("Cách 4", dedup_rebuild), ("Cách 5", dedup_sort)]:
    h = build_list(arr); t = time.perf_counter(); r = f(h); dt = (time.perf_counter() - t) * 1000
    print(f"{name}: còn {len(to_list(r))} nút ({dt:.0f} ms)")
h = build_list(arr[:2000]); t = time.perf_counter(); dedup_bruteforce(h); print(f"Cách 1 (n=2000): {(time.perf_counter()-t)*1000:.0f} ms")

Cách 2: còn 36887 nút (30 ms)


Cách 3: còn 36887 nút (14 ms)


Cách 4: còn 36887 nút (33 ms)
Cách 5: còn 36887 nút (37 ms)


Cách 1 (n=2000): 96 ms


## 12. Bộ test

Đối chiếu mọi cách với một bản tham chiếu viết trên **mảng** (`[v for v in a if a.count(v) == 1]`) trên danh sách ngẫu nhiên (seed cố định), miền giá trị nhỏ để có nhiều trùng. Kiểm tra thêm: các nút trả về là **chính các nút gốc** (xóa tại chỗ, không tạo nút mới) với Cách 2/3/5.

In [17]:
def reference(arr):
    return [v for v in arr if arr.count(v) == 1]

INPLACE = {"counter": dedup_counter, "array": SolutionArray().deleteDuplicatesUnsorted, "sort": dedup_sort,
           "submit": Solution().deleteDuplicatesUnsorted, "brute": dedup_bruteforce}
def run_all(arr):
    exp = reference(arr)
    for k, f in INPLACE.items():
        head = build_list(arr)
        ids = set()
        cur = head
        while cur: ids.add(id(cur)); cur = cur.next
        res = f(head)
        assert to_list(res) == exp, (k, arr)
        cur = res
        while cur:
            assert id(cur) in ids, (k, "nút mới được tạo!"); cur = cur.next
    assert to_list(dedup_rebuild(build_list(arr))) == exp
    return exp

for arr, exp in examples:
    assert run_all(arr) == exp
edge = [([1], [1]), ([1, 1], []), ([1, 2], [1, 2]), ([5, 5, 5, 5], []), ([1, 2, 3, 4], [1, 2, 3, 4]),
        ([7, 1, 2, 7], [1, 2]), ([1, 7, 2, 2], [1, 7]), ([100000, 1, 100000], [1])]
for arr, exp in edge:
    assert run_all(arr) == exp, arr
rng = random.Random(18360)
for _ in range(600):
    arr = [rng.randint(1, rng.choice([2, 5, 20, 10 ** 5])) for _ in range(rng.randint(1, 30))]
    run_all(arr)
print("Ví dụ, edge case (xóa ở đầu/cuối/toàn bộ), 600 test ngẫu nhiên: OK")

Ví dụ, edge case (xóa ở đầu/cuối/toàn bộ), 600 test ngẫu nhiên: OK


### 12.1 Kiểm tra tính chất

Với mọi input, kết quả phải: (1) là **dãy con** (giữ thứ tự) của danh sách gốc; (2) chỉ chứa giá trị có tần suất đúng 1 trong danh sách gốc; (3) chứa **mọi** giá trị như vậy; (4) mọi giá trị trong kết quả khác nhau.

In [18]:
def is_subsequence(sub, arr):
    it = iter(arr)
    return all(any(x == y for y in it) for x in sub)

rng = random.Random(77)
for _ in range(300):
    arr = [rng.randint(1, 8) for _ in range(rng.randint(1, 20))]
    res = to_list(Solution().deleteDuplicatesUnsorted(build_list(arr)))
    f = Counter(arr)
    assert is_subsequence(res, arr)
    assert all(f[v] == 1 for v in res) and set(res) == {v for v in f if f[v] == 1} and len(set(res)) == len(res)
print("4 tính chất đúng trên 300 danh sách ngẫu nhiên")

4 tính chất đúng trên 300 danh sách ngẫu nhiên


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Giữ lại **một** bản của giá trị trùng (nhầm với "remove duplicates")

In [19]:
def wrong_keep_one(head):
    seen = set(); dummy = ListNode(0, head); prev, cur = dummy, head
    while cur:
        if cur.val in seen: prev.next = cur.next          # SAI: chỉ xóa các lần xuất hiện sau
        else: seen.add(cur.val); prev = cur
        cur = cur.next
    return dummy.next

arr, exp = examples[0]
print("sai :", to_list(wrong_keep_one(build_list(arr))), "| đúng:", exp)

sai : [1, 2, 3] | đúng: [1, 3]


**Vì sao sai:** đề yêu cầu xóa **mọi** nút có giá trị lặp, kể cả lần xuất hiện đầu tiên. **Sửa:** đếm tần suất trước, xóa nút có `freq > 1`.

### Lỗi 2 — Di chuyển `prev` cả khi vừa xóa

In [20]:
def wrong_move_prev(head):
    freq = Counter(to_list(head)); dummy = ListNode(0, head); prev, cur = dummy, head
    while cur:
        if freq[cur.val] > 1:
            prev.next = cur.next
        prev = cur                                         # SAI: prev trỏ vào nút đã bị xóa
        cur = cur.next
    return dummy.next

arr, exp = examples[2]
print("sai :", to_list(wrong_move_prev(build_list(arr))), "| đúng:", exp)

sai : [2, 1, 2, 4] | đúng: [1, 4]


**Vì sao sai:** sau khi xóa `cur`, `prev = cur` làm `prev` trỏ vào nút **đã tách khỏi** danh sách; lần xóa tiếp theo sửa `next` của nút "ma" đó — không ảnh hưởng danh sách thật ⇒ các nút trùng liền nhau còn sót. **Sửa:** chỉ `prev = cur` khi **giữ** nút.

### Lỗi 3 — Không dùng dummy, quên xử lý nút đầu bị xóa

In [21]:
def wrong_no_dummy(head):
    freq = Counter(to_list(head))
    prev, cur = head, head.next                            # SAI: giả định head luôn được giữ
    while cur:
        if freq[cur.val] > 1: prev.next = cur.next
        else: prev = cur
        cur = cur.next
    return head

arr, exp = examples[1]
print("sai :", to_list(wrong_no_dummy(build_list(arr))), "| đúng:", exp)

sai : [2] | đúng: []


**Vì sao sai:** nút đầu (2) cũng phải xóa nhưng code luôn trả `head`. **Sửa:** dùng `dummy = ListNode(0, head)` và trả `dummy.next`.

### Lỗi 4 — Xóa trong **một** lượt dựa trên các giá trị đã gặp (không biết tương lai)

In [22]:
def wrong_one_pass(head):
    seen = Counter(); dummy = ListNode(0, head); prev, cur = dummy, head
    while cur:
        seen[cur.val] += 1
        if seen[cur.val] > 1: prev.next = cur.next      # SAI: lần đầu tiên chưa biết sẽ trùng
        else: prev = cur
        cur = cur.next
    return dummy.next

arr, exp = examples[2]
print("sai :", to_list(wrong_one_pass(build_list(arr))), "| đúng:", exp)

sai : [3, 2, 1, 4] | đúng: [1, 4]


**Vì sao sai:** ở nút `3` đầu tiên, số đếm mới là 1 nên bị giữ, dù về sau còn `3` khác. Đây là lý do bắt buộc phải có lượt đếm **trước**. **Sửa:** hai lượt.

### Lỗi 5 — Đếm trong lúc đang xóa (dùng chung một lượt sửa danh sách để đếm lại)

Nếu đếm bằng cách duyệt danh sách **đã bị sửa dở** (như vét cạn đếm lại mà không đánh dấu trước), kết quả sai: sau khi xóa bản thứ nhất của `2`, bản còn lại chỉ đếm được 1 ⇒ bị giữ.

In [23]:
def wrong_count_while_deleting(head):
    dummy = ListNode(0, head); prev, cur = dummy, head
    while cur:
        c, p = 0, dummy.next
        while p:
            c += p.val == cur.val; p = p.next            # SAI: đếm trên danh sách đã bị xóa bớt
        if c > 1: prev.next = cur.next
        else: prev = cur
        cur = cur.next
    return dummy.next

arr, exp = examples[0]
print("sai :", to_list(wrong_count_while_deleting(build_list(arr))), "| đúng:", exp)

sai : [1, 3, 2] | đúng: [1, 3]


**Vì sao sai:** tần suất phải đo trên danh sách **gốc**. **Sửa:** đếm xong toàn bộ (hoặc đánh dấu) rồi mới xóa.

### Lỗi 6 — Viết đệ quy trên danh sách dài 10⁵ nút

In [24]:
def dedup_recursive(head, freq=None):
    if freq is None:
        freq = count_values_todo(head) if 'count_values_todo' in globals() else Counter(to_list(head))
    if head is None:
        return None
    rest = dedup_recursive(head.next, freq)
    if freq[head.val] > 1:
        return rest
    head.next = rest
    return head

print("danh sách ngắn:", to_list(dedup_recursive(build_list([3, 2, 2, 1, 3, 2, 4]))))
try:
    dedup_recursive(build_list(list(range(1, 5001))))
    print("không lỗi")
except RecursionError as e:
    print("danh sách 5000 nút -> RecursionError:", type(e).__name__)
print("bản lặp vẫn chạy:", len(to_list(Solution().deleteDuplicatesUnsorted(build_list(list(range(1, 5001)))))), "nút")

danh sách ngắn: [1, 4]
danh sách 5000 nút -> RecursionError: RecursionError
bản lặp vẫn chạy: 5000 nút


**Vì sao sai:** đệ quy sâu bằng số nút (`10⁵`) vượt giới hạn đệ quy mặc định của Python (1000). **Sửa:** dùng vòng lặp với `prev`/`cur` (Cách 2).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Quyết định xóa phụ thuộc **toàn cục** ⇒ **hai lượt**: lượt đếm, lượt sửa.
- Sửa danh sách liên kết ⇒ luôn nghĩ tới **dummy** + `prev`.
- Khi xóa: **không** dịch `prev`; khi giữ: `prev = cur`.
- Giá trị bị chặn nhỏ ⇒ mảng đếm thay `Counter`.
- Đọc kỹ: "xóa mọi bản sao" (bài này, 82) khác "giữ một bản" (83, 1836 khác 83).

## 15. Biến thể của bài

1. **83 Remove Duplicates from Sorted List**: danh sách đã sắp, giữ một bản — một lượt.
2. **82 Remove Duplicates from Sorted List II**: đã sắp, xóa mọi bản — một lượt nhờ các bản trùng nằm liền nhau.
3. **Giữ một bản trong danh sách chưa sắp**: một lượt với `set` (Lỗi 1 chính là lời giải của biến thể này!).
4. **Xóa giá trị xuất hiện đúng `k` lần / ít nhất `k` lần** (Bài tập 5).
5. **203 Remove Linked List Elements**: xóa theo một giá trị cho trước — cùng kỹ thuật dummy.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đếm lại mỗi nút | Đánh dấu rồi xóa | `O(n²)` | `O(n)` | Kiểm chứng |
| 2. Counter + dummy | Hai lượt | `O(n)` | `O(u)` | **Bản nộp** |
| 3. Mảng đếm | Như 2, không băm | `O(n + V)` | `O(V)` | Giá trị nhỏ, cần nhanh |
| 4. Dựng lại | Mảng → lọc → list mới | `O(n)` | `O(n)` | Khi được tạo nút mới |
| 5. Sắp tìm trùng | Không hash | `O(n log n)` | `O(n)` | Cấm dùng hash table |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `[1,1,2]`; (b) `[4,5,6]`; (c) `[9,8,9,8,7]`; (d) `[1,2,1,3,2,4,3,5]`.

**Bài 2 (tính tay):** Trace lượt 2 cho `[5,1,5,2,1,3]`: ghi `cur`, hành động, `prev`.

**Bài 3 (điền TODO):** Hoàn thiện `count_values(head)` trả `Counter` giá trị của danh sách.

**Bài 4 (tự viết lại):** Viết `keep_first_unsorted(head)` cho biến thể **giữ một bản** (lần xuất hiện đầu) trong danh sách chưa sắp, sửa tại chỗ.

**Bài 5 (mini challenge):** Viết `delete_freq_at_least(head, k)` xóa mọi nút có giá trị xuất hiện **ít nhất `k` lần** (`k ≥ 1`), và `delete_freq_exactly(head, k)` xóa mọi nút có giá trị xuất hiện **đúng `k` lần`.

In [25]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_values_todo(head):
    freq = Counter()
    # TODO: duyệt cur từ head, tăng freq[cur.val]
    return freq

In [26]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def keep_first_unsorted(head):
    # TODO: một lượt, set đã gặp, dummy + prev
    pass

def delete_freq_at_least(head, k):
    # TODO: như bản nộp với điều kiện freq >= k
    pass

def delete_freq_exactly(head, k):
    # TODO: điều kiện freq == k
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: đếm tần suất. Mức 2: giữ giá trị xuất hiện 1 lần. Mức 3: (a) `[2]`; (b) `[4,5,6]`; (c) `[7]`; (d) `[4,5]`.

**Bài 2.** Mức 1: `freq = {5:2, 1:2, 2:1, 3:1}`. Mức 2: xóa 5, xóa 1, xóa 5, giữ 2, xóa 1, giữ 3. Mức 3: kết quả `[2, 3]`.

**Bài 3.** Mức 1: `cur = head`. Mức 2: `while cur`. Mức 3: `freq[cur.val] += 1; cur = cur.next`.

**Bài 4.** Mức 1: `seen = set()`. Mức 2: đã gặp ⇒ xóa, chưa ⇒ thêm vào `seen` và giữ. Mức 3: đối chiếu `list(dict.fromkeys(arr))`.

**Bài 5.** Mức 1: hai lượt như bài gốc. Mức 2: đổi điều kiện. Mức 3: `delete_freq_at_least(head, 2)` chính là bài gốc; `k = 1` xóa hết.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1, 2, 3
b1 = [to_list(Solution().deleteDuplicatesUnsorted(build_list(a))) for a in [[1, 1, 2], [4, 5, 6], [9, 8, 9, 8, 7], [1, 2, 1, 3, 2, 4, 3, 5]]]
print("Bài 1:", b1); assert b1 == [[2], [4, 5, 6], [7], [4, 5]]
print("Bài 2:"); assert trace_counter([5, 1, 5, 2, 1, 3]) == [2, 3]
def count_values_todo(head):
    freq = Counter(); cur = head
    while cur:
        freq[cur.val] += 1; cur = cur.next
    return freq
assert count_values_todo(build_list([3, 2, 2, 1])) == Counter({2: 2, 3: 1, 1: 1})
print("Bài 3: OK")

Bài 1: [[2], [4, 5, 6], [7], [4, 5]]
Bài 2:
freq = {5: 2, 1: 2, 2: 1, 3: 1}
bước 1: cur=5 (freq 2) -> xóa | prev=dummy | danh sách: [1, 5, 2, 1, 3]
bước 2: cur=1 (freq 2) -> xóa | prev=dummy | danh sách: [5, 2, 1, 3]
bước 3: cur=5 (freq 2) -> xóa | prev=dummy | danh sách: [2, 1, 3]
bước 4: cur=2 (freq 1) -> giữ | prev=2 | danh sách: [2, 1, 3]
bước 5: cur=1 (freq 2) -> xóa | prev=2 | danh sách: [2, 3]
bước 6: cur=3 (freq 1) -> giữ | prev=3 | danh sách: [2, 3]
Bài 3: OK


In [28]:
# Bài 4, 5
def keep_first_unsorted(head):
    seen = set(); dummy = ListNode(0, head); prev, cur = dummy, head
    while cur:
        if cur.val in seen: prev.next = cur.next
        else: seen.add(cur.val); prev = cur
        cur = cur.next
    return dummy.next

def _delete_if(head, pred):
    freq = count_values_todo(head); dummy = ListNode(0, head); prev, cur = dummy, head
    while cur:
        if pred(freq[cur.val]): prev.next = cur.next
        else: prev = cur
        cur = cur.next
    return dummy.next

def delete_freq_at_least(head, k):
    return _delete_if(head, lambda f: f >= k)

def delete_freq_exactly(head, k):
    return _delete_if(head, lambda f: f == k)

rng = random.Random(5)
for _ in range(400):
    arr = [rng.randint(1, 5) for _ in range(rng.randint(1, 15))]; k = rng.randint(1, 4)
    assert to_list(keep_first_unsorted(build_list(arr))) == list(dict.fromkeys(arr))
    assert to_list(delete_freq_at_least(build_list(arr), k)) == [v for v in arr if arr.count(v) < k]
    assert to_list(delete_freq_exactly(build_list(arr), k)) == [v for v in arr if arr.count(v) != k]
    assert to_list(delete_freq_at_least(build_list(arr), 2)) == reference(arr)
print("Bài 4, 5: OK |", to_list(keep_first_unsorted(build_list([3, 2, 2, 1, 3, 2, 4]))), to_list(delete_freq_exactly(build_list([3, 2, 2, 1, 3, 2, 4]), 2)))

Bài 4, 5: OK | [3, 2, 1, 4] [2, 2, 1, 2, 4]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đếm rồi xóa trên danh sách liên kết":**

```python
freq = Counter(); cur = head
while cur: freq[cur.val] += 1; cur = cur.next
dummy = ListNode(0, head); prev, cur = dummy, head
while cur:
    if freq[cur.val] > 1: prev.next = cur.next   # xóa: prev đứng yên
    else:                 prev = cur             # giữ: prev tiến
    cur = cur.next
return dummy.next
```

**Checklist:**
- [ ] Xóa **mọi** bản của giá trị lặp (không giữ bản đầu)?
- [ ] Đếm trên danh sách **gốc** trước khi xóa (hai lượt)?
- [ ] Dùng **dummy**, trả `dummy.next` (nút đầu có thể bị xóa, danh sách có thể rỗng)?
- [ ] Chỉ dịch `prev` khi **giữ** nút?
- [ ] Đã test: xóa toàn bộ, xóa nút đầu, xóa nút cuối, các bản trùng liền nhau?